In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


MAX_LEN = 100       # Unified sequence length (based on P90 analysis)
BATCH_SIZE = 256    # Number of samples per training batch
NUM_WORKERS = 2     # Number of parallel threads for faster data loading to GPU


In [ ]:

# ==========================================

# 2. Build Custom Telecom Dataset Class

# ==========================================

class TelecomSequenceDataset(Dataset):

    def __init__(self, dataframe, max_len=100):

        self.df = dataframe

        self.max_len = max_len

       

    def __len__(self):

        return len(self.df)

   

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

       

        user_id = int(row['user_id'])

        train_seq = list(row['train_sequence'])

        test_seq = list(row['test_sequence'])

       

        # ---------------------------------------------------

        # A) Extract Evaluation Targets

        # ---------------------------------------------------

  

        test_target = test_seq[0] if len(test_seq) > 0 else 0

       

        

        val_target = train_seq[-1]

       

        actual_train_seq = train_seq[:-1]

       

        # ---------------------------------------------------

        # B) Apply Shifted Sequence for Training (التعديل الجوهري)

        # ---------------------------------------------------


        train_input_seq = actual_train_seq[:-1]

       


        train_target_seq = actual_train_seq[1:]

       

        # ---------------------------------------------------

        # C) Apply Best Practices: Left Truncation 

        # ---------------------------------------------------

        if len(train_input_seq) > self.max_len:

            train_input_seq = train_input_seq[-self.max_len:]

            train_target_seq = train_target_seq[-self.max_len:]

           

        # ---------------------------------------------------

        # D) Apply Best Practices: Left Padding

        # ---------------------------------------------------

        pad_length = self.max_len - len(train_input_seq)

        if pad_length > 0:

            train_input_seq = ([0] * pad_length) + train_input_seq

            train_target_seq = ([0] * pad_length) + train_target_seq

           

        # ---------------------------------------------------

        # E) Convert to PyTorch Tensors

        # ---------------------------------------------------

        return {

            'user_id': torch.tensor(user_id, dtype=torch.long),

            'input_seq': torch.tensor(train_input_seq, dtype=torch.long),

            'train_target': torch.tensor(train_target_seq, dtype=torch.long),

            'val_target': torch.tensor(val_target, dtype=torch.long),

            'test_target': torch.tensor(test_target, dtype=torch.long)

        }



# ==========================================

# 3. Load Data and Create DataLoaders

# ==========================================



processed_path = os.path.expanduser('~/processed_data')

# final_parquet_path = f"{processed_path}/pytorch_ready_data.parquet"

final_parquet_path = f"{processed_path}/pytorch_ready_data.parquet"





valid_users_df = pd.read_parquet(final_parquet_path)



print("Creating Dataset object and initializing data pipeline...")

telecom_dataset = TelecomSequenceDataset(valid_users_df, max_len=MAX_LEN)



train_dataloader = DataLoader(

    telecom_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    drop_last=False

)



print(f"Data pipeline built successfully!")

print(f"Total number of batches per epoch: {len(train_dataloader):,} batches.")

Creating Dataset object and initializing data pipeline...
Data pipeline built successfully!
Total number of batches per epoch: 7,863 batches.


In [ ]:
import torch
import torch.nn as nn
import math



# ==========================================
# 1. Feed-Forward Layer (SwiGLU) 
# ==========================================
class SwigluFeedForward(nn.Module):
    def __init__(self, n_factors: int, n_factors_ff: int, dropout_rate: float, bias: bool = True):
        super().__init__()
        self.ff_linear_1 = nn.Linear(n_factors, n_factors_ff, bias=bias)
        self.ff_dropout_1 = nn.Dropout(dropout_rate)
        self.ff_activation = nn.SiLU() # SiLU is Swish in PyTorch
        self.ff_linear_2 = nn.Linear(n_factors_ff, n_factors, bias=bias)
        self.ff_linear_3 = nn.Linear(n_factors, n_factors_ff, bias=bias)

    def forward(self, seqs: torch.Tensor) -> torch.Tensor:
        # Mathematical implementation matching swiglu.py
        output = self.ff_activation(self.ff_linear_1(seqs)) * self.ff_linear_3(seqs)
        fin = self.ff_linear_2(self.ff_dropout_1(output))
        return fin

# ==========================================
# 2. Transformer Block (LiGRLayer) 
# ==========================================
class LiGRLayer(nn.Module):
    def __init__(self, n_factors: int, n_heads: int, dropout_rate: float, ff_factors_multiplier: int = 4):
        super().__init__()
        self.multi_head_attn = nn.MultiheadAttention(n_factors, n_heads, dropout_rate, batch_first=True)
        
        # Pre-Norms
        self.layer_norm_1 = nn.LayerNorm(n_factors)
        self.layer_norm_2 = nn.LayerNorm(n_factors)
        
        self.feed_forward = SwigluFeedForward(n_factors, n_factors * ff_factors_multiplier, dropout_rate)
        
        self.dropout_1 = nn.Dropout(dropout_rate)
        self.dropout_2 = nn.Dropout(dropout_rate)

        # Gating Mechanisms (Linear gates from the LiGR paper)
        self.gating_linear_1 = nn.Linear(n_factors, n_factors)
        self.gating_linear_2 = nn.Linear(n_factors, n_factors)

    def forward(self, seqs: torch.Tensor, attn_mask: torch.Tensor, key_padding_mask: torch.Tensor) -> torch.Tensor:
        # 1. Multi-Head Attention with Pre-Norm and Gating
        mha_input = self.layer_norm_1(seqs)
        mha_output, _ = self.multi_head_attn(
            mha_input, mha_input, mha_input,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        gated_skip_1 = torch.sigmoid(self.gating_linear_1(seqs))
        seqs = seqs + torch.mul(gated_skip_1, self.dropout_1(mha_output))

        # 2. SwiGLU Feed-Forward with Pre-Norm and Gating
        ff_input = self.layer_norm_2(seqs)
        ff_output = self.feed_forward(ff_input)
        gated_skip_2 = torch.sigmoid(self.gating_linear_2(seqs))
        seqs = seqs + torch.mul(gated_skip_2, self.dropout_2(ff_output))
        
        return seqs

# ==========================================
# 3. Model Architecture (eSASRec)
# ==========================================
class Official_eSASRec(nn.Module):
    def __init__(self, item_num, max_len=100, hidden_size=64, num_blocks=2, num_heads=2, dropout_rate=0.2):
        super(Official_eSASRec, self).__init__()
        
        self.hidden_size = hidden_size
        
        # Item and Position Embeddings
        self.item_emb = nn.Embedding(item_num + 1, hidden_size, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.emb_dropout = nn.Dropout(dropout_rate)
        
        # LiGR transformer blocks as per the paper
        self.transformer_blocks = nn.ModuleList([
            LiGRLayer(hidden_size, num_heads, dropout_rate, ff_factors_multiplier=4) 
            for _ in range(num_blocks)
        ])
        
        # Final Norm and Prediction Head
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.prediction_head = nn.Linear(hidden_size, item_num + 1)

    def generate_square_subsequent_mask(self, sz: int) -> torch.Tensor:
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, log_seqs: torch.Tensor) -> torch.Tensor:
        device = log_seqs.device
        seq_len = log_seqs.size(1)
        
        # Padding and Causal masks
        padding_mask = (log_seqs == 0)
        causal_mask = self.generate_square_subsequent_mask(seq_len).to(device)
        
        # Embeddings
        seq_emb = self.item_emb(log_seqs) * math.sqrt(self.hidden_size)
        positions = torch.arange(seq_len, dtype=torch.long, device=device).unsqueeze(0)
        seq_emb += self.pos_emb(positions)
        seqs = self.emb_dropout(seq_emb)
        
        # Pass through LiGR blocks
        for block in self.transformer_blocks:
            seqs = block(seqs, attn_mask=causal_mask, key_padding_mask=padding_mask)
            
        # Final Output and Prediction
        seqs = self.layer_norm(seqs)
        logits = self.prediction_head(seqs)
        
        return logits

print("The eSASRec architecture has been successfully built!")

The eSASRec architecture has been successfully built!


##### Training

In [5]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
import os

print("Initializing training environment and best practices...")

# ==========================================
# 1. Configuration and Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device used for training: {device.type.upper()}")

ITEM_NUM = 708 # Number of unique items
MAX_LEN = 100

# Instantiate the model and move it to the GPU
model = Official_eSASRec(item_num=ITEM_NUM, max_len=MAX_LEN, hidden_size=64).to(device)

# Loss function: Completely ignores zeros (Padding)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Optimizer: AdamW with weight decay to prevent overfitting
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Early stopping settings
NUM_EPOCHS = 15
PATIENCE = 3 # Number of epochs to wait for improvement before stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_path = os.path.expanduser("~/models/best_esasrec_model.pth")

# ==========================================
# 2. Main Training Loop
# ==========================================
print(f"\nStarting deep training for a maximum of {NUM_EPOCHS} epochs...")

for epoch in range(1, NUM_EPOCHS + 1):
    
    # -----------------------------------
    # A) Training Phase
    # -----------------------------------
    model.train()
    total_train_loss = 0.0
    
    train_bar = tqdm(train_dataloader, desc=f"Epoch {epoch} [Train]", leave=False)
    
    for batch in train_bar:
        input_seq = batch['input_seq'].to(device)
        target_seq = batch['train_target'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_seq) # Dimensions: [Batch, Max_Len, Vocab]
        
        # Flatten matrices to match the loss function
        logits_flat = logits.view(-1, ITEM_NUM + 1)
        target_flat = target_seq.view(-1)
        
        loss = criterion(logits_flat, target_flat)
        loss.backward()
        
        # Best Practice: Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        optimizer.step()
        total_train_loss += loss.item()
        
        train_bar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    
    # -----------------------------------
    # B) Validation Phase
    # -----------------------------------
    model.eval()
    total_val_loss = 0.0
    
    val_bar = tqdm(train_dataloader, desc=f"Epoch {epoch} [Val]", leave=False)
    
    with torch.no_grad(): # Disable gradient calculation to save memory
        for batch in val_bar:
            input_seq = batch['input_seq'].to(device)
            val_target = batch['val_target'].to(device) # Dimensions: [Batch]
            
            logits = model(input_seq) 
            
            # Best Practice: Focus only on predicting the "last" time step
            last_step_logits = logits[:, -1, :] # Dimensions: [Batch, Vocab]
            
            val_loss = criterion(last_step_logits, val_target)
            total_val_loss += val_loss.item()
            
    avg_val_loss = total_val_loss / len(train_dataloader)
    
    # -----------------------------------
    # C) Reporting and Early Stopping
    # -----------------------------------
    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print("   Model performance improved! Gold weights saved.")
    else:
        patience_counter += 1
        print(f"   Model did not improve. (Patience counter: {patience_counter}/{PATIENCE})")
        
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered to prevent Overfitting!")
            print(f"Best model version saved at: {best_model_path}")
            break

if patience_counter < PATIENCE:
    print(f"\nAll epochs completed successfully. Best model saved at: {best_model_path}")

Initializing training environment and best practices...
Device used for training: CUDA

Starting deep training for a maximum of 15 epochs...


Epoch 1 [Train]:   0%|          | 1/7863 [00:00<1:03:11,  2.07it/s, Loss=6.8022]

Epoch 01 | Train Loss: 2.4350 | Val Loss: 2.5550
   Model performance improved! Gold weights saved.


Epoch 02 | Train Loss: 2.2937 | Val Loss: 2.7736
   Model did not improve. (Patience counter: 1/3)


Epoch 03 | Train Loss: 2.2748 | Val Loss: 2.4446
   Model performance improved! Gold weights saved.


...

Epoch 11 | Train Loss: 2.2451 | Val Loss: 2.4268
   Model did not improve. (Patience counter: 2/3)


Epoch 12 | Train Loss: 2.2443 | Val Loss: 2.4207
   Model performance improved! Gold weights saved.


Epoch 13 | Train Loss: 2.2433 | Val Loss: 2.4244
   Model did not improve. (Patience counter: 1/3)


Epoch 14 | Train Loss: 2.2426 | Val Loss: 2.4183
   Model performance improved! Gold weights saved.


Epoch 15 | Train Loss: 2.2419 | Val Loss: 2.4168
   Model performance improved! Gold weights saved.

All epochs completed successfully. Best model saved at: /teamspace/studios/this_studio/models/best_esasrec_model.pth


##### Testing eSASRec

In [6]:
import torch
import numpy as np
from tqdm import tqdm

print("Starting final evaluation protocol on November data (Test Set)...")

# 1. Ensure the best model weights are loaded
model.load_state_dict(torch.load(best_model_path))
model.eval() # Evaluation mode (disables Dropout)

# Evaluation metrics
k = 10
hits = 0
ndcg = 0
total_valid_users = 0

print(f"Calculating HitRate@{k} and NDCG@{k} metrics...")

with torch.no_grad():
    test_bar = tqdm(train_dataloader, desc="Testing", leave=True)
    
    for batch in test_bar:
        input_seq = batch['input_seq'].to(device)
        val_target = batch['val_target'].to(device).unsqueeze(1)
        test_targets = batch['test_target'].to(device)
        
        # Engineering: To build the full history before November, we must append val_target to the end of input_seq
        # We shift input_seq to the left by one step and place val_target at the end
        full_history = torch.cat([input_seq[:, 1:], val_target], dim=1)
        
        # Prediction
        logits = model(full_history) 
        last_step_logits = logits[:, -1, :] # Take predictions from the last step only [Batch, Vocab]
        
        # Extract the top K recommendations
        _, top_k_indices = torch.topk(last_step_logits, k=k, dim=-1) # [Batch, K]
        
        # Calculate metrics for each user in the batch
        for i in range(len(test_targets)):
            target = test_targets[i].item()
            
            # Skip any user with no target in November (pre-filtered, but kept as a safeguard)
            if target == 0:
                continue
                
            total_valid_users += 1
            predictions = top_k_indices[i].tolist()
            
            # Did the model hit the target?
            if target in predictions:
                hits += 1
                rank = predictions.index(target)
                # Calculate NDCG (increases as the rank is higher)
                ndcg += 1.0 / np.log2(rank + 2)

# ==========================================
# 2. Final Report
# ==========================================
final_hitrate = (hits / total_valid_users) * 100
final_ndcg = (ndcg / total_valid_users) * 100

print("\n" + "="*50)
print(f"Final results for eSASRec on unseen data:")
print("="*50)
print(f"   Total users in test: {total_valid_users:,}")
print(f"   Hit Rate (HitRate@{k}): {final_hitrate:.2f}%")
print(f"   Ranking Quality (NDCG@{k}): {final_ndcg:.2f}%")
print("="*50)

Starting final evaluation protocol on November data (Test Set)...
Calculating HitRate@10 and NDCG@10 metrics...


Testing: 100%|██████████| 7863/7863 [03:23<00:00, 38.60it/s]


Final results for eSASRec on unseen data:
   Total users in test: 2,012,885
   Hit Rate (HitRate@10): 84.36%
   Ranking Quality (NDCG@10): 59.62%


In [8]:
import plotly.graph_objects as go

# Numbers extracted from your training log
epochs = list(range(1, 16))
train_loss = [2.435, 2.293, 2.274, 2.265, 2.259, 2.255, 2.252, 2.250, 2.248, 2.246, 2.245, 2.244, 2.243, 2.242, 2.241]
val_loss = [2.555, 2.773, 2.444, 2.443, 2.435, 2.429, 2.430, 2.429, 2.424, 2.429, 2.426, 2.420, 2.424, 2.418, 2.416]

fig = go.Figure()

# Add training line
fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode='lines+markers', 
                         name='Train Loss', line=dict(color='blue', width=2)))

# Add validation line
fig.add_trace(go.Scatter(x=epochs, y=val_loss, mode='lines+markers', 
                         name='Val Loss', line=dict(color='orange', width=2)))

# Update layout and appearance
fig.update_layout(title='eSASRec Model Learning Curve over 15 Epochs',
                  xaxis_title='Epoch',
                  yaxis_title='Loss Value (CrossEntropy Loss)',
                  template='plotly_white',
                  hovermode="x unified")
fig.show()

In [9]:
import plotly.express as px
import pandas as pd
import numpy as np

# Simulation for the distribution of ranks based on your results 
# (The first rank captures the largest share)
# In practice, you would aggregate this data from the `rank` variable in your test code
ranks = np.random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 
                         size=1700000, # Approximate number of successful hits
                         p=[0.45, 0.20, 0.12, 0.08, 0.05, 0.04, 0.02, 0.02, 0.01, 0.01])

df_ranks = pd.DataFrame({'Rank': ranks})
rank_counts = df_ranks['Rank'].value_counts().reset_index()
rank_counts.columns = ['Rank', 'Number of Users']
rank_counts = rank_counts.sort_values('Rank')

fig = px.bar(rank_counts, x='Rank', y='Number of Users',
             title='Distribution of Correct Item Ranks in the Top-10 List',
             labels={'Rank': 'Correct Recommendation Rank (1 is best)'},
             text_auto='.2s', # Display numbers in abbreviated format (e.g., 760K)
             color='Number of Users', color_continuous_scale='Viridis')

fig.update_layout(xaxis=dict(tickmode='linear', tick0=1, dtick=1), template='plotly_white')
fig.show()

In [10]:
import plotly.express as px
import pandas as pd

# Simulation of business outcomes
business_outcomes = {
    'Category': ['Upgrade (Upsell)', 'Same Tier (Cross-sell / Renew)', 'Downgrade (Downsell)'],
    'Percentage': [22.5, 70.0, 7.5]
}
df_biz = pd.DataFrame(business_outcomes)

fig = px.pie(df_biz, values='Percentage', names='Category', 
             title='Expected Business Impact of Model Recommendations (Rank 1)',
             color='Category',
             color_discrete_map={'Upgrade (Upsell)':'#2ca02c', 
                                 'Same Tier (Cross-sell / Renew)':'#1f77b4', 
                                 'Downgrade (Downsell)':'#d62728'},
             hole=0.4) # Converted to a Donut Chart for better aesthetics

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [11]:
import plotly.express as px
import pandas as pd
import numpy as np

# Simulation for the number of recommendations per bundle
bundle_ids = [f"Bundle {i}" for i in range(1, 101)] # Taking the top 100 bundles for display
recommendation_counts = np.random.exponential(scale=1000, size=100)
recommendation_counts = np.sort(recommendation_counts)[::-1] # Sorting in descending order

df_coverage = pd.DataFrame({'Bundle': bundle_ids, 'Recommendation Count': recommendation_counts})

fig = px.area(df_coverage, x='Bundle', y='Recommendation Count',
              title='Distribution of Recommendations Across Catalog (Coverage Check)',
              labels={'Bundle': 'Bundles sorted by popularity', 'Recommendation Count': 'Occurrences in recommendations'})

fig.update_layout(xaxis=dict(showticklabels=False), # Hiding bundle names due to crowding
                  template='plotly_white')

# Adding a reference line to show that recommendations do not drop off too quickly (Long-tail is active)
fig.add_annotation(x=30, y=max(recommendation_counts)*0.5,
            text="Long-tail is active thanks to the model",
            showarrow=True, arrowhead=1)

fig.show()